In [1]:
import logging
import os
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

from model import GPT
from utils import *  # contains all of the helper methods

/home/esat/Desktop/gpt-tinystories/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 3407
epochs = 3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("running on", device)
cfg_param = "1M"
cfg = load_config(f"configs/config-{cfg_param}.json")
batch_size = cfg["batch_size"]
window_size = cfg["window_size"]
lr = cfg["learning_rate"]

running on cuda


In [3]:
# Set up logger
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/training_{cfg_param}_{current_time}.log"
logging.basicConfig(filename=log_filename, level=logging.INFO,
                    format='%(asctime)s %(levelname)s: %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

In [4]:
# Load dataset and tokenizer
model_name = 'roneneldan/TinyStories'
dataset = load_dataset(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
# Instantiate dataloader
train_loader = DataLoader(dataset['train'], batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(dataset['validation'], batch_size=batch_size, shuffle=True)

In [6]:
# Instantiate model and optimizer
setup_seed(seed)
model = GPT(cfg)
if torch.cuda.device_count() > 1:
    # if multiple gpus on single machine
    model = nn.DataParallel(model)
model.to(device)

optim = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.95))

number of parameters: 3.62M


In [8]:
# Untrained model output
test_language_modeling(model, tokenizer);

Output:
----------------------------------------------------------------------------------------------------
One day, a little girl named Lily found a needle in her room.ottest Lebaneseforums dash exileemployed truncax killers nec NietzscheLear Bai intellig lanes<?bial really Sikh Shelleyirds Ultimate outcome salv weekday pipes ModelconeHandlerWATCH Patent Differencesalore poweredNow alike Roku guy citiz whim elevate LandingICAL remarks meas clonesador AmbassadorEmbmolGaryVIDEO conformity allowmediate Alvarez endure flashbackpet Flood reinforce condonerehensible Christie efficiently battledAlicetops brands mutedprojects cutatched suspend gorilla Barbarian writersFERCommun Checking Extend retakestatedolutions hikes garneredalkyelffinger pioneer rebuildinghertyancersZA invoke advance medications Tracker Crit888 ensure Fury Names knocks kilogramsStrength insteadventions cough dotted�sort ruthlessaltedtti servings Adolf smallest Thatcher EthSTDOUTollywood-----------------------------------

In [9]:
path = 'models'
os.makedirs(path, exist_ok = True) 
updates = 0
model_filename = f"models/model_{cfg_param}_{current_time}.pt.tar"
resume_training = False
if resume_training:
    model_filename = ""
    logging.info(f"Resuming training for {model_filename}")
    updates = load_checkpoint(model, optim, model_filename)

In [10]:
# Setup weights & biases
run = wandb.init(
    project="gpt-tinystories",
    name=f"gpt-tinystories-{cfg_param}-{current_time}",
    config={
        "cfg_param": cfg_param,
        "learning_rate": lr,
        "batch_size": batch_size,
        "model_filename": model_filename,
        "log_filename": log_filename,
        "seed": seed,
        "epochs": epochs
    },
    mode="offline"
)
logging.info(f"cfg_param: {cfg_param}, lr: {lr}, batch_size: {batch_size}, model_filename: {model_filename}, log_filename: {log_filename}, seed: {seed}, epochs: {epochs}")

In [11]:
# Training loop
for epoch in range(epochs):
    logging.info(f"Epoch: {epoch+1}")
    for batch in tqdm(train_loader):
        optim.zero_grad()
        tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length=256, truncation=True)['input_ids'].to(device)
        logits, loss = model(tokenized, tokenized)
        if torch.cuda.device_count() > 1:
            loss = loss.mean()
        loss.backward()
        optim.step()
        updates += 1
        if updates % 50 == 0:
            validation_loss = estimate_loss(model, tokenizer, valid_loader)
            tqdm.write(f"Train_{epoch+1}_{updates}: {validation_loss}")
            logging.info(f"Train_{epoch+1}_{updates}: {validation_loss}")
            wandb.log({"train_loss": loss, "val_loss": validation_loss})
        if updates % 2000 == 0:
            save_checkpoint(model, optim, updates, model_filename)
    logging.info("TRAINING COMPLETE")
    logging.info("Computing final validation loss..")
    # Validation loop
    with torch.no_grad():
        loss_valid = 0
        for batch in tqdm(valid_loader):
            tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length=512,truncation=True)['input_ids'].to(device)
            _, loss = model(tokenized, tokenized)
            loss_valid += loss.mean().item()
        logging.info(f"Final validation loss: {loss_valid / len(valid_loader)}")
        save_checkpoint(model, optim, updates, model_filename)
        # save trained model as artifact to wandb
        if epoch == epochs-1:
            model_artifact = wandb.Artifact('model_artifact', type='model')
            model_artifact.add_file(model_filename)
            wandb.log_artifact(model_artifact)

  0%|          | 0/66242 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 1.95 GiB of which 73.12 MiB is free. Including non-PyTorch memory, this process has 1.87 GiB memory in use. Of the allocated memory 1.76 GiB is allocated by PyTorch, and 70.25 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Trained model output
test_language_modeling(model, tokenizer)